# CrackSegDiff: Simplified Training & Inference
Automated setup, data preparation (First 500 Test / 2000 Train), training, and testing.

In [ ]:
# 1. Setup Environment & Weights
!nvidia-smi
import os
if not os.path.exists('CrackSegDiff'):
    !git clone https://github.com/Ludwig-H/CrackSegDiff.git
%cd CrackSegDiff
!git pull
!pip install -r requirement.txt

# Force install pre-built wheels for Mamba-SSM and Causal-Conv1d to avoid compilation errors and ensure GPU speed
print("Installing Optimized Mamba Kernels...")
import torch
cuda_version = torch.version.cuda.replace('.', '')
torch_version = torch.__version__.split('+')[0].replace('.', '')
# Assuming standard Colab PyTorch 2.x and CUDA 11.8 or 12.x
# We use the releases from Dao-AILab which are reliable
!pip install ninja  # Speeds up compilation
!pip install causal-conv1d>=1.0.0 --no-build-isolation -v
!pip install mamba-ssm>=1.0.1 --no-build-isolation -v

# If the above standard install fails (it tries to build), we could try finding wheels:
# (But usually --no-build-isolation helps or just standard pip works if env is clean)

# Verify installation
try:
    import mamba_ssm
    print("\nSUCCESS: Mamba SSM installed successfully! GPU acceleration enabled (x10 speed).")
except ImportError:
    print("\nWARNING: Mamba SSM installation failed.")
    print("Fallback to Pure Python active (SLOWER).")

# Download Pretrained Weights
!mkdir -p pretrained_weights
!gdown 1JYqMxM5dbCLZ-WGPKtIofYJhj0VPuy3l -O pretrained_weights/vssm_base_0229_ckpt_epoch_237.pth

# Patch Hardcoded Paths
target_file = 'CrackSegDiff/guided_diffusion/unet.py'
new_path = os.path.abspath('pretrained_weights/vssm_base_0229_ckpt_epoch_237.pth')
if os.path.exists(target_file):
    with open(target_file, 'r') as f: content = f.read()
    content = content.replace('/home/dell/jlc/segdiff/pre_trained_weights/vssm_base_0229_ckpt_epoch_237.pth', new_path)
    with open(target_file, 'w') as f: f.write(content)
    print("Path patched successfully.")

In [ ]:
# 2. Prepare Data (First 500 Test / 2000 Train)
!rm -rf data && mkdir -p data
%cd data
!gdown 1qnLMCeon7LJjT9H0ENiNF5sFs-F7-NvK -O data.zip
!unzip -q -o data.zip
%cd ..

import os, glob, shutil
print("Organizing Data...")

# Find folders
try:
    src_img = glob.glob('data/**/img/fused', recursive=True)[0]
    src_lbl = glob.glob('data/**/lbs', recursive=True)[0]
except IndexError:
    # Fallback if 'fused' not found, try 'intensity'
    print("Fused folder not found, checking intensity...")
    src_img = glob.glob('data/**/img/intensity', recursive=True)[0]
    src_lbl = glob.glob('data/**/lbs', recursive=True)[0]

print(f"Images source: {src_img}")
print(f"Labels source: {src_lbl}")

# Get sorted file lists
img_files = sorted(glob.glob(os.path.join(src_img, '*')))
lbl_files = sorted(glob.glob(os.path.join(src_lbl, '*.bmp')))

# Split: First 500 Test, Rest Train
test_pairs = list(zip(img_files[:500], lbl_files[:500]))
train_pairs = list(zip(img_files[500:], lbl_files[500:]))

print(f"Test Set: {len(test_pairs)} (First 500)")
print(f"Train Set: {len(train_pairs)} (Rest)")

# Copy to formatted directories
train_dir = os.path.abspath('data/Train')
test_dir = os.path.abspath('data/Test')

for pairs, dest in [(train_pairs, train_dir), (test_pairs, test_dir)]:
    os.makedirs(os.path.join(dest, '5d'), exist_ok=True)
    os.makedirs(os.path.join(dest, 'mask'), exist_ok=True)
    for img, mask in pairs:
        shutil.copy(img, os.path.join(dest, '5d'))
        shutil.copy(mask, os.path.join(dest, 'mask'))
print("Data preparation complete.")

In [ ]:
# 3. Train Model
import os
data_dir = os.path.abspath('data/Train')
out_dir = os.path.abspath('results/train_output')
os.makedirs(out_dir, exist_ok=True)

!python CrackSegDiff/segmentation_train.py --data_dir {data_dir} --out_dir {out_dir} --image_size 256 --num_channels 128 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 1000 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --lr 5e-5 --batch_size 8 --save_interval 5000

In [ ]:
# 4. Inference
import glob, os
# Use latest trained model
models = sorted(glob.glob('results/train_output/*.pt'))
model_path = models[-1] if models else "pretrained_weights/savedmodel100000.pt"
test_dir = os.path.abspath('data/Test')
print(f"Using model: {model_path}")

for modality in ['intensity', 'range', 'fused']:
    out_path = f"results/test_output_{modality}"
    os.makedirs(out_path, exist_ok=True)
    print(f"Testing {modality}...")
    !python CrackSegDiff/segmentation_sample.py --data_dir {test_dir} --out_dir {out_path} --model_path {model_path} --modality {modality} --image_size 256 --num_channels 128 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 1000 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --num_ensemble 1

In [ ]:
# 5. Zip Results
!zip -r results.zip results